# 🇮🇳 Indian Portfolio Dataset Builder v2.0
**Improvements over v1:** Full Nifty 50 universe · OHLCV fields · 100+ engineered features · Macro & FX context · Sectoral benchmarks · Fill-flag quality column · CVaR efficient frontier · Data quality report

---

## 1. Install & Import

In [ ]:
!pip install -q cvxpy clarabel yfinance pandas_ta requests lxml

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import cvxpy as cp
import pandas_ta as ta
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import requests
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries loaded.')

## 2. Universe Definition
Pulls the current Nifty 50 constituent list live from Wikipedia, then appends `.NS` for NSE tickers.

In [ ]:
# --- 2a. Nifty 50 constituents (scraped live) ---
try:
    nifty_tables = pd.read_html('https://en.wikipedia.org/wiki/NIFTY_50')
    # Find the table that has a 'Symbol' column
    nifty_df = next(t for t in nifty_tables if 'Symbol' in t.columns)
    nifty50_tickers = [s.strip().replace(' ', '') + '.NS' for s in nifty_df['Symbol'].tolist()]
    print(f'✅ Scraped {len(nifty50_tickers)} Nifty 50 tickers from Wikipedia.')
except Exception as e:
    print(f'⚠️  Wikipedia scrape failed ({e}). Using hardcoded fallback list.')
    nifty50_tickers = [
        'ADANIENT.NS','ADANIPORTS.NS','APOLLOHOSP.NS','ASIANPAINT.NS','AXISBANK.NS',
        'BAJAJ-AUTO.NS','BAJFINANCE.NS','BAJAJFINSV.NS','BEL.NS','BHARTIARTL.NS',
        'BPCL.NS','BRITANNIA.NS','CIPLA.NS','COALINDIA.NS','DIVISLAB.NS',
        'DRREDDY.NS','EICHERMOT.NS','GRASIM.NS','HCLTECH.NS','HDFCBANK.NS',
        'HDFCLIFE.NS','HEROMOTOCO.NS','HINDALCO.NS','HINDUNILVR.NS','ICICIBANK.NS',
        'INDUSINDBK.NS','INFY.NS','ITC.NS','JSWSTEEL.NS','KOTAKBANK.NS',
        'LT.NS','M&M.NS','MARUTI.NS','NESTLEIND.NS','NTPC.NS',
        'ONGC.NS','POWERGRID.NS','RELIANCE.NS','SBILIFE.NS','SBIN.NS',
        'SHRIRAMFIN.NS','SUNPHARMA.NS','TATACONSUM.NS','TATAMOTORS.NS','TATASTEEL.NS',
        'TCS.NS','TECHM.NS','TITAN.NS','TRENT.NS','ULTRACEMCO.NS','WIPRO.NS'
    ]

# --- 2b. Hedge instruments ---
hedge_tickers = [
    'GOLDBEES.NS',    # Gold ETF (domestic hedge)
    'LIQUIDBEES.NS',  # Liquid / cash-equivalent ETF
    'CPSEETF.NS',     # PSU Equity ETF
]

# --- 2c. Benchmarks ---
benchmark_tickers = [
    '^NSEI',          # Nifty 50
    '^NSEBANK',       # Nifty Bank
    '^CNXPHARMA',     # Nifty Pharma
    '^CNXMETAL',      # Nifty Metal
    '^CNXAUTO',       # Nifty Auto
    '^CNXIT',         # Nifty IT
]

# --- 2d. Macro / FX tickers ---
macro_tickers = [
    'USDINR=X',       # USD/INR spot rate
    'EURINR=X',       # EUR/INR spot rate
    'CL=F',           # Crude oil (WTI) — key driver of Indian inflation
    'GC=F',           # Gold futures (cross-check with GOLDBEES)
    '^INDIAVIX',      # India VIX (fear index)
]

all_tickers = nifty50_tickers + hedge_tickers + benchmark_tickers + macro_tickers
print(f'📋 Total tickers queued for download: {len(all_tickers)}')

## 3. Download OHLCV Data

In [ ]:
START = '2018-01-01'
END   = '2026-03-24'

print(f'🚀 Downloading OHLCV from {START} to {END} for {len(all_tickers)} tickers...')
raw = yf.download(
    all_tickers,
    start=START,
    end=END,
    auto_adjust=True,   # adjusts for splits & dividends automatically
    progress=True
)

print(f'\n📦 Raw shape: {raw.shape}  (rows × [fields × tickers])')
print(f'   Fields available: {list(raw.columns.get_level_values(0).unique())}')

## 4. Data Quality: Forward-Fill with Flag Column

In [ ]:
# Cap ffill at 3 consecutive stale days (more = suspect data, not just weekend)
MAX_FFILL = 3

close_raw  = raw['Close'].copy()
volume_raw = raw['Volume'].copy()

# Build is_ffilled flag BEFORE filling (True = row was stale/filled)
close_filled_flag  = close_raw.isnull()
close_data         = close_raw.ffill(limit=MAX_FFILL)

# After capped ffill, any remaining NaN means ticker didn't exist yet — drop those
# Keep tickers that have at least 80% coverage
coverage = close_data.notna().mean()
good_tickers = coverage[coverage >= 0.80].index.tolist()
dropped      = [t for t in close_data.columns if t not in good_tickers]

print(f'✅ Tickers with ≥80% coverage: {len(good_tickers)}')
if dropped:
    print(f'⚠️  Dropped ({len(dropped)} tickers with sparse history): {dropped}')

close_data  = close_data[good_tickers].dropna(how='all')
volume_data = volume_raw[good_tickers].ffill(limit=MAX_FFILL).fillna(0)
fill_flag   = close_filled_flag[good_tickers].reindex(close_data.index).fillna(False)

print(f'\n📅 Date range: {close_data.index[0].date()} → {close_data.index[-1].date()}')
print(f'   Rows: {len(close_data)}   Tickers: {len(close_data.columns)}')

## 5. Data Quality Report

In [ ]:
print('=' * 60)
print('DATA QUALITY REPORT')
print('=' * 60)

# Per-ticker summary
quality_df = pd.DataFrame({
    'first_date'    : close_data.apply(lambda c: c.first_valid_index()),
    'last_date'     : close_data.apply(lambda c: c.last_valid_index()),
    'coverage_pct'  : (close_data.notna().mean() * 100).round(1),
    'ffill_days'    : fill_flag.sum().astype(int),
    'ffill_pct'     : (fill_flag.mean() * 100).round(2),
    'mean_price'    : close_data.mean().round(2),
    'std_price'     : close_data.std().round(2),
})
quality_df = quality_df.sort_values('coverage_pct', ascending=False)

print(quality_df.to_string())

# Overall stats
print('\n--- Summary ---')
print(f'Total trading days : {len(close_data)}')
print(f'Total tickers      : {len(close_data.columns)}')
print(f'Total fill events  : {fill_flag.values.sum():,}')
print(f'Avg fill rate      : {fill_flag.mean().mean()*100:.2f}%')

# Save quality report
quality_df.to_csv('data_quality_report.csv')
print('\n✅ Saved: data_quality_report.csv')

## 6. Feature Engineering
Price-based signals for each equity/ETF ticker.

In [ ]:
# Work on the equity + hedge tickers (exclude benchmarks and macro for features)
equity_and_hedge = [t for t in good_tickers if t in nifty50_tickers + hedge_tickers]

feature_frames = []

print(f'⚙️  Engineering features for {len(equity_and_hedge)} tickers...')

for ticker in equity_and_hedge:
    if ticker not in close_data.columns:
        continue

    c = close_data[ticker].dropna()
    v = volume_data[ticker].reindex(c.index).fillna(0)

    # --- Build per-ticker OHLCV (we have adjusted close; reconstruct proxy H/L/O) ---
    # Best effort: use raw multi-index if available
    try:
        o = raw['Open'][ticker].reindex(c.index).ffill()
        h = raw['High'][ticker].reindex(c.index).ffill()
        l = raw['Low'][ticker].reindex(c.index).ffill()
    except Exception:
        o = c; h = c; l = c

    df = pd.DataFrame({'open': o, 'high': h, 'low': l, 'close': c, 'volume': v}, index=c.index)

    # --- Returns ---
    df['ret_1d']   = c.pct_change(1)
    df['ret_5d']   = c.pct_change(5)
    df['ret_21d']  = c.pct_change(21)
    df['ret_63d']  = c.pct_change(63)
    df['log_ret']  = np.log(c / c.shift(1))

    # --- Volatility ---
    df['vol_5d']   = df['log_ret'].rolling(5).std()  * np.sqrt(252)
    df['vol_21d']  = df['log_ret'].rolling(21).std() * np.sqrt(252)
    df['vol_63d']  = df['log_ret'].rolling(63).std() * np.sqrt(252)

    # --- Moving averages ---
    df['sma_20']  = c.rolling(20).mean()
    df['sma_50']  = c.rolling(50).mean()
    df['sma_200'] = c.rolling(200).mean()
    df['ema_12']  = c.ewm(span=12, adjust=False).mean()
    df['ema_26']  = c.ewm(span=26, adjust=False).mean()

    # --- Momentum & trend ---
    df['momentum_21']  = c / c.shift(21) - 1
    df['momentum_63']  = c / c.shift(63) - 1
    df['momentum_252'] = c / c.shift(252) - 1
    df['above_sma200'] = (c > df['sma_200']).astype(int)
    df['price_to_52wh']= c / c.rolling(252).max()   # Price as % of 52-week high
    df['price_to_52wl']= c / c.rolling(252).min()   # Price as % of 52-week low

    # --- MACD ---
    df['macd']         = df['ema_12'] - df['ema_26']
    df['macd_signal']  = df['macd'].ewm(span=9, adjust=False).mean()
    df['macd_hist']    = df['macd'] - df['macd_signal']

    # --- RSI ---
    delta  = c.diff()
    gain   = delta.clip(lower=0).rolling(14).mean()
    loss   = (-delta.clip(upper=0)).rolling(14).mean()
    rs     = gain / loss.replace(0, np.nan)
    df['rsi_14'] = 100 - (100 / (1 + rs))

    # --- Bollinger Bands ---
    bb_mid         = c.rolling(20).mean()
    bb_std         = c.rolling(20).std()
    df['bb_upper'] = bb_mid + 2 * bb_std
    df['bb_lower'] = bb_mid - 2 * bb_std
    df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / bb_mid
    df['bb_pct']   = (c - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])

    # --- ATR (Average True Range) ---
    tr = pd.concat([
        h - l,
        (h - c.shift(1)).abs(),
        (l - c.shift(1)).abs()
    ], axis=1).max(axis=1)
    df['atr_14'] = tr.rolling(14).mean()

    # --- Volume signals ---
    df['vol_ma20']    = v.rolling(20).mean()
    df['vol_ratio']   = v / df['vol_ma20'].replace(0, np.nan)  # Volume surge ratio
    df['obv']         = (np.sign(df['ret_1d']) * v).cumsum()   # On-Balance Volume
    # VWAP proxy (rolling 20-day)
    typical_price     = (h + l + c) / 3
    df['vwap_20d']    = (typical_price * v).rolling(20).sum() / v.rolling(20).sum().replace(0, np.nan)

    # --- Drawdown ---
    roll_max          = c.cummax()
    df['drawdown']    = c / roll_max - 1
    df['max_dd_63d']  = df['drawdown'].rolling(63).min()

    # --- Skewness & Kurtosis of returns ---
    df['skew_21d']    = df['log_ret'].rolling(21).skew()
    df['kurt_21d']    = df['log_ret'].rolling(21).kurt()

    # --- Fill flag ---
    df['is_ffilled']  = fill_flag[ticker].reindex(c.index).astype(int)

    # --- Multi-index column labeling ---
    df.columns = pd.MultiIndex.from_product([[ticker], df.columns])
    feature_frames.append(df)

# Combine all tickers
features_df = pd.concat(feature_frames, axis=1)
print(f'\n✅ Feature matrix shape: {features_df.shape}')
print(f'   Features per ticker  : {len(features_df.columns.get_level_values(1).unique())}')
print(f'   Feature list         : {list(features_df.columns.get_level_values(1).unique())}')

## 7. Macro & FX Context Layer

In [ ]:
macro_cols = [t for t in good_tickers if t in macro_tickers]
bench_cols = [t for t in good_tickers if t in benchmark_tickers]

macro_df = close_data[macro_cols].copy() if macro_cols else pd.DataFrame(index=close_data.index)
bench_df = close_data[bench_cols].copy() if bench_cols else pd.DataFrame(index=close_data.index)

# Rename for clarity
rename_map = {
    'USDINR=X': 'USDINR', 'EURINR=X': 'EURINR',
    'CL=F': 'Crude_WTI', 'GC=F': 'Gold_Futures',
    '^INDIAVIX': 'India_VIX',
    '^NSEI': 'Nifty50', '^NSEBANK': 'NiftyBank',
    '^CNXPHARMA': 'NiftyPharma', '^CNXMETAL': 'NiftyMetal',
    '^CNXAUTO': 'NiftyAuto', '^CNXIT': 'NiftyIT',
}
macro_df = macro_df.rename(columns=rename_map)
bench_df = bench_df.rename(columns=rename_map)

# Returns for macro series
macro_ret_df = macro_df.pct_change().add_suffix('_ret')
bench_ret_df = bench_df.pct_change().add_suffix('_ret')

# RBI Repo Rate from FRED (free, no API key needed for public series)
try:
    fred_url = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=INDIRLTLT01STM'
    rbi_rate = pd.read_csv(fred_url, index_col=0, parse_dates=True)
    rbi_rate.columns = ['RBI_rate_pct']
    rbi_rate = rbi_rate.resample('B').ffill()  # Business-day frequency
    macro_df  = macro_df.join(rbi_rate, how='left').ffill()
    print('✅ RBI interest rate series fetched from FRED.')
except Exception as e:
    print(f'⚠️  FRED fetch skipped ({e}). RBI rate not included.')

# Combine all macro into one frame aligned to equity index
macro_combined = pd.concat([macro_df, macro_ret_df, bench_df, bench_ret_df], axis=1)
macro_combined = macro_combined.reindex(close_data.index).ffill(limit=5)

print(f'✅ Macro frame shape: {macro_combined.shape}')
print(f'   Columns: {list(macro_combined.columns)}')

## 8. Assemble & Save the Master Dataset

In [ ]:
# --- 8a. Flat close-price matrix (easy to use) ---
price_matrix = close_data[equity_and_hedge].copy()
price_matrix.to_csv('Indian_Portfolio_Prices_v2.csv')
print(f'✅ Price matrix saved:    Indian_Portfolio_Prices_v2.csv  {price_matrix.shape}')

# --- 8b. Macro + benchmark series ---
macro_combined.to_csv('Indian_Portfolio_Macro_v2.csv')
print(f'✅ Macro data saved:      Indian_Portfolio_Macro_v2.csv   {macro_combined.shape}')

# --- 8c. Per-ticker features in long format (ML-friendly) ---
# Stack to long: index=[Date, Ticker], columns=features
long_df = features_df.stack(level=0)
long_df.index.names = ['Date', 'Ticker']
long_df = long_df.reset_index()
long_df.to_csv('Indian_Portfolio_Features_Long_v2.csv', index=False)
print(f'✅ Long features saved:   Indian_Portfolio_Features_Long_v2.csv  {long_df.shape}')

# --- 8d. Wide features (Date as index, MultiIndex columns) ---
features_df.to_csv('Indian_Portfolio_Features_Wide_v2.csv')
print(f'✅ Wide features saved:   Indian_Portfolio_Features_Wide_v2.csv  {features_df.shape}')

print('\n📊 Dataset inventory:')
print(f'   Tickers in feature set : {len(equity_and_hedge)}')
print(f'   Features per ticker    : {len(features_df.columns.get_level_values(1).unique())}')
print(f'   Trading days           : {len(long_df["Date"].unique())}')
print(f'   Total feature rows     : {len(long_df):,}')

## 9. Returns & Correlation Analysis

In [ ]:
returns = price_matrix.pct_change().dropna()

# Annualised stats
ann_ret  = returns.mean() * 252
ann_vol  = returns.std()  * np.sqrt(252)
sharpe   = ann_ret / ann_vol
max_dd   = (price_matrix / price_matrix.cummax() - 1).min()

stats = pd.DataFrame({
    'Ann_Return_%' : (ann_ret * 100).round(2),
    'Ann_Vol_%'    : (ann_vol * 100).round(2),
    'Sharpe'       : sharpe.round(3),
    'Max_Drawdown_%': (max_dd * 100).round(2),
}).sort_values('Sharpe', ascending=False)

print('Annualised Asset Statistics (sorted by Sharpe):')
print(stats.to_string())

# Correlation heatmap
corr = returns.corr()
fig, ax = plt.subplots(figsize=(16, 13))
im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
labels = [t.replace('.NS','') for t in corr.columns]
ax.set_xticklabels(labels, rotation=90, fontsize=7)
ax.set_yticklabels(labels, fontsize=7)
ax.set_title('Nifty 50 Correlation Matrix (daily returns, 2018–2026)', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: correlation_matrix.png')

## 10. CVaR Optimization — Efficient Frontier
Sweeps across a grid of target returns to build the full CVaR frontier, not just a single point.

In [ ]:
# --- 10a. Define the optimization function ---
def optimize_cvar(ret_data: pd.DataFrame,
                  alpha: float = 0.05,
                  target_ret: float = 0.0008,
                  allow_short: bool = False):
    """
    CVaR-C (Conditional Value at Risk - Constrained) portfolio optimizer.
    Returns (weights, cvar_value, portfolio_return) or (None, None, None) if infeasible.
    """
    n_assets    = ret_data.shape[1]
    n_scenarios = ret_data.shape[0]
    w           = cp.Variable(n_assets)
    zeta        = cp.Variable()
    u           = cp.Variable(n_scenarios)
    mu          = np.mean(ret_data.values, axis=0)

    cvar_obj = zeta + (1 / (n_scenarios * alpha)) * cp.sum(u)
    constraints = [
        u >= 0,
        u >= -ret_data.values @ w - zeta,
        cp.sum(w) == 1,
        mu @ w >= target_ret,
    ]
    if not allow_short:
        constraints.append(w >= 0)          # Long-only
    else:
        constraints.append(w >= -0.10)      # Max 10% short per asset
        constraints.append(w <= 0.40)       # Max 40% per asset long

    prob = cp.Problem(cp.Minimize(cvar_obj), constraints)
    prob.solve(solver=cp.CLARABEL, verbose=False)

    if prob.status in ['optimal', 'optimal_inaccurate'] and w.value is not None:
        port_ret  = float(mu @ w.value)
        port_cvar = float(prob.value)
        return w.value, port_cvar, port_ret
    return None, None, None


# --- 10b. Select assets for optimization ---
# Use the tickers that are in the Nifty 50 + hedge and have good data
opt_assets = [t for t in equity_and_hedge if t in returns.columns]
ret_opt    = returns[opt_assets].dropna()

daily_min  = ret_opt.mean().min()
daily_max  = ret_opt.mean().max()
target_grid = np.linspace(daily_min * 0.5, daily_max * 0.85, 40)

print(f'🔍 Optimizing across {len(target_grid)} target-return levels...')
print(f'   Assets: {len(opt_assets)}   Scenarios: {len(ret_opt)}')

frontier_results = []
alpha_levels = [0.01, 0.05, 0.10]   # 3 risk-confidence levels

for alpha in alpha_levels:
    print(f'  α = {alpha}...', end=' ')
    for tgt in target_grid:
        w_val, cvar_val, port_ret = optimize_cvar(ret_opt, alpha=alpha, target_ret=tgt)
        if w_val is not None:
            row = {
                'alpha'      : alpha,
                'target_ret' : tgt,
                'port_ret_ann': port_ret * 252,
                'cvar_ann'   : cvar_val * np.sqrt(252),
                'sharpe_est' : (port_ret * 252) / (ret_opt[opt_assets].std().mean() * np.sqrt(252)),
            }
            for i, asset in enumerate(opt_assets):
                row[asset] = round(float(w_val[i]), 6)
            frontier_results.append(row)
    print('done.')

frontier_df = pd.DataFrame(frontier_results)
frontier_df.to_csv('CVaR_Efficient_Frontier.csv', index=False)
print(f'\n✅ Saved: CVaR_Efficient_Frontier.csv  ({len(frontier_df)} frontier points)')

## 11. Optimal Portfolios — Long-Only vs. Long/Short

In [ ]:
# Solve for a target return near the 60th percentile of the frontier
med_target = np.percentile(target_grid, 60)

results = {}
for mode, allow_short in [('Long-Only', False), ('Long/Short', True)]:
    w_val, cvar_val, port_ret = optimize_cvar(ret_opt, alpha=0.05,
                                               target_ret=med_target,
                                               allow_short=allow_short)
    if w_val is not None:
        results[mode] = pd.Series(dict(zip(opt_assets, np.round(w_val, 4))))
        print(f'\n--- {mode} Portfolio (α=5%, daily target={med_target:.5f}) ---')
        df_w = results[mode].sort_values(ascending=False)
        df_w = df_w[df_w.abs() > 0.001]  # Drop near-zero weights
        print(df_w.to_string())
        print(f'CVaR: {cvar_val:.6f}  |  Port Return: {port_ret:.6f}/day  |  Ann Return: {port_ret*252:.2%}')

# Save weights
weights_df = pd.DataFrame(results)
weights_df.to_csv('Optimal_CVaR_Weights_v2.csv')
print('\n✅ Saved: Optimal_CVaR_Weights_v2.csv')

## 12. Visualisations

In [ ]:
# --- 12a. CVaR Efficient Frontier (all alpha levels) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = {0.01: '#E24B4A', 0.05: '#BA7517', 0.10: '#3B6D11'}
for alpha, grp in frontier_df.groupby('alpha'):
    grp_sorted = grp.sort_values('cvar_ann')
    axes[0].plot(grp_sorted['cvar_ann'] * 100,
                 grp_sorted['port_ret_ann'] * 100,
                 label=f'α={alpha}', color=colors.get(alpha, 'gray'), linewidth=2)
axes[0].set_xlabel('Annualised CVaR (%)', fontsize=11)
axes[0].set_ylabel('Annualised Portfolio Return (%)', fontsize=11)
axes[0].set_title('CVaR Efficient Frontier — 3 Confidence Levels', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- 12b. Optimal Long-Only weight bar chart ---
if 'Long-Only' in results:
    w_plot = results['Long-Only'].sort_values(ascending=False)
    w_plot = w_plot[w_plot > 0.005]
    bar_labels = [t.replace('.NS', '') for t in w_plot.index]
    bar_vals   = w_plot.values * 100
    bar_colors = ['#3B6D11' if v > 10 else '#BA7517' if v > 5 else '#B4B2A9' for v in bar_vals]
    axes[1].barh(bar_labels[::-1], bar_vals[::-1], color=bar_colors[::-1], edgecolor='none')
    axes[1].set_xlabel('Allocation (%)', fontsize=11)
    axes[1].set_title('Optimal Long-Only CVaR Weights (α=5%)', fontsize=12)
    axes[1].grid(True, alpha=0.3, axis='x')
    for i, v in enumerate(bar_vals[::-1]):
        axes[1].text(v + 0.2, i, f'{v:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('frontier_and_weights.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: frontier_and_weights.png')

In [ ]:
# --- 12c. Macro context panel ---
macro_plot_cols = [c for c in macro_combined.columns
                   if c in ['USDINR', 'Crude_WTI', 'India_VIX', 'Nifty50']]

if len(macro_plot_cols) >= 2:
    fig, axes = plt.subplots(len(macro_plot_cols), 1, figsize=(14, 3 * len(macro_plot_cols)), sharex=True)
    if len(macro_plot_cols) == 1:
        axes = [axes]

    palette = ['#185FA5', '#D85A30', '#639922', '#534AB7']
    for i, col in enumerate(macro_plot_cols):
        s = macro_combined[col].dropna()
        axes[i].plot(s.index, s.values, color=palette[i % len(palette)], linewidth=1)
        axes[i].set_ylabel(col, fontsize=10)
        axes[i].grid(True, alpha=0.25)
    axes[-1].set_xlabel('Date', fontsize=10)
    fig.suptitle('Macro Context Series (2018–2026)', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig('macro_context.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved: macro_context.png')
else:
    print('⚠️  Not enough macro columns available to plot.')

In [ ]:
# --- 12d. Feature distribution check: RSI and Volatility for a sample ticker ---
sample_ticker = opt_assets[0] if opt_assets else None

if sample_ticker and (sample_ticker, 'rsi_14') in features_df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    name = sample_ticker.replace('.NS', '')

    axes[0].hist(features_df[(sample_ticker, 'rsi_14')].dropna(), bins=50,
                 color='#378ADD', edgecolor='none', alpha=0.8)
    axes[0].set_title(f'{name} — RSI(14) distribution', fontsize=11)
    axes[0].axvline(30, color='#E24B4A', lw=1.5, linestyle='--', label='Oversold 30')
    axes[0].axvline(70, color='#3B6D11', lw=1.5, linestyle='--', label='Overbought 70')
    axes[0].legend(fontsize=9)

    axes[1].plot(features_df.index,
                 features_df[(sample_ticker, 'vol_21d')].values,
                 color='#D85A30', linewidth=0.8)
    axes[1].set_title(f'{name} — 21-day Annualised Vol', fontsize=11)
    axes[1].set_ylabel('Volatility')

    axes[2].plot(features_df.index,
                 features_df[(sample_ticker, 'drawdown')].values,
                 color='#A32D2D', linewidth=0.8)
    axes[2].fill_between(features_df.index,
                          features_df[(sample_ticker, 'drawdown')].values,
                          0, color='#A32D2D', alpha=0.2)
    axes[2].set_title(f'{name} — Drawdown from peak', fontsize=11)

    plt.tight_layout()
    plt.savefig('feature_check.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved: feature_check.png')

## 13. Final Output Summary

In [ ]:
print('=' * 65)
print('DATASET BUILD COMPLETE — FILE SUMMARY')
print('=' * 65)

outputs = [
    ('Indian_Portfolio_Prices_v2.csv',       'Adjusted close prices — all tickers'),
    ('Indian_Portfolio_Macro_v2.csv',        'Macro + benchmark + FX series'),
    ('Indian_Portfolio_Features_Long_v2.csv','Feature matrix (long format, ML-ready)'),
    ('Indian_Portfolio_Features_Wide_v2.csv','Feature matrix (wide, MultiIndex cols)'),
    ('CVaR_Efficient_Frontier.csv',          'Full frontier — 3 alpha levels × 40 targets'),
    ('Optimal_CVaR_Weights_v2.csv',          'Optimal Long-Only & Long/Short weights'),
    ('data_quality_report.csv',              'Per-ticker quality: coverage, ffill stats'),
    ('correlation_matrix.png',               'Heatmap of daily return correlations'),
    ('frontier_and_weights.png',             'CVaR frontier + optimal weight bar chart'),
    ('macro_context.png',                    'Macro time-series panel'),
    ('feature_check.png',                    'Sample ticker feature sanity plots'),
]

import os
for fname, desc in outputs:
    exists = '✅' if os.path.exists(fname) else '❌ MISSING'
    size   = f'({os.path.getsize(fname)/1024:.0f} KB)' if os.path.exists(fname) else ''
    print(f'  {exists}  {fname:<45} {size:<10}  {desc}')

print('\n📈 Dataset Stats:')
print(f'  Equity + hedge tickers : {len(equity_and_hedge)}')
print(f'  Benchmark series       : {len(bench_cols)}')
print(f'  Macro series           : {len(macro_cols)}')
print(f'  Features per ticker    : {len(features_df.columns.get_level_values(1).unique())}')
print(f'  Trading days           : {len(returns)}')
print(f'  Total ML feature rows  : {len(long_df):,}')
print(f'  Frontier points        : {len(frontier_df)}')
print('\n🏁 Done.')